In [ ]:
##**IEM 4723 -- Assignment 3**
##**Andrew Gungoll**
##**A20367050**

In [3]:
%pip install redis pymysql pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 4.2 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import re, importlib.util, inspect
import pandas as pd
import pymysql
import redis

CWID = "A20367050"   
MYSQL_HOST = "host.docker.internal"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "OSUGoPokes"
REDIS_HOST = "host.docker.internal"
REDIS_PORT = 6380
REDIS_DB = 0         

DATA_DIR = Path("data")
if not (DATA_DIR / "orders" / "Customer.csv").exists():
    matches = list(Path.cwd().rglob("Customer.csv"))
    if matches:
        DATA_DIR = matches[0].parent.parent
print("Data folder:", DATA_DIR.resolve())
assert (DATA_DIR / "orders" / "Customer.csv").exists(), "Set DATA_DIR to the supplied data folder"

r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, db=REDIS_DB, decode_responses=True)
print("Redis ping:", r.ping())
connection = pymysql.connect(host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_USER,
                             password=MYSQL_PASSWORD, autocommit=True)
print("MySQL connected")

Data folder: /home/jovyan/iem4723/Assignment 3/data
Redis ping: True
MySQL connected


In [4]:
##Part 1 Question 1
customers = pd.read_csv(DATA_DIR / "orders" / "Customer.csv")
orders = pd.read_csv(DATA_DIR / "orders" / "CustomerOrder.csv")

print("Customer rows:", len(customers))
print("CustomerOrder rows:", len(orders))

print("\nCUSTOMER COLUMNS:")
print(customers.dtypes)
print("\nCUSTOMER SAMPLE:")
display(customers.head())

print("\nORDER COLUMNS:")
print(orders.dtypes)
print("\nORDER SAMPLE:")
display(orders.head())

Customer rows: 10
CustomerOrder rows: 9

CUSTOMER COLUMNS:
Customer_Num          object
Customer_FirstName    object
Customer_LastName     object
Customer_Address      object
Customer_City         object
Customer_State        object
Customer_ZIP           int64
Customer_Phone         int64
Customer_Email        object
dtype: object

CUSTOMER SAMPLE:


,Customer_Num,Customer_FirstName,Customer_LastName,Customer_Address,Customer_City,Customer_State,Customer_ZIP,Customer_Phone,Customer_Email
0,C25,Bob,Felton,100 CRYSTAL A DRIVE,HERSHEY,PA,17033,2482604235,Bob@Felton.com
1,C2501,John,Grant,225 WEST STATION SQUARE DRIVE,PITTSBURGH,PA,15219,7817993378,John@grant.com
2,C35,Jill,Young,500 FRANK W BURR BOULEVARD,TEANECK,NJ,7666,9035970279,Jill@Young.com
3,C493,Russ,Jackson,500 WEST MADISON STREET,CHICAGO,IL,60661,8788658690,Russ@Young.com
4,C658,Chris,George,7500 DALLAS PARKWAY,PLANO,TX,75024,9778250256,Chris@George.com



ORDER COLUMNS:
Order_Num                int64
Customer_Num            object
Order_Date              object
Order_ReqShipDate       object
Order_ActualShipDate    object
Order_Carrier           object
dtype: object

ORDER SAMPLE:


,Order_Num,Customer_Num,Order_Date,Order_ReqShipDate,Order_ActualShipDate,Order_Carrier
0,16639375,C2501,2021-11-19,2021-11-19,2021-11-19,*** Customer Pickup ***
1,16642826,C25,2021-10-12,2021-10-12,2021-10-13,DHL Ground
2,16644074,C25,2021-10-25,2021-10-25,2021-10-25,UPS Ground
3,16645282,C35,2021-11-07,2021-11-07,2021-11-20,DHL Ground
4,16656274,C493,2021-11-08,2021-11-08,2021-11-08,FedEX FFL


In [7]:
DB_NAME = f"redis_compare_{CWID}"

with connection.cursor() as cursor:
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}`")
    cursor.execute(f"USE `{DB_NAME}`")

print("Database created:", DB_NAME)

Database created: redis_compare_A20367050


In [8]:
with connection.cursor() as cursor:

    cursor.execute("DROP TABLE IF EXISTS customerorder")
    cursor.execute("DROP TABLE IF EXISTS customer")

    cursor.execute("""
        CREATE TABLE customer (
            Customer_Num VARCHAR(20) PRIMARY KEY,
            Customer_FirstName VARCHAR(100),
            Customer_LastName VARCHAR(100),
            Customer_Address VARCHAR(255),
            Customer_City VARCHAR(100),
            Customer_State VARCHAR(10),
            Customer_ZIP VARCHAR(10),
            Customer_Phone VARCHAR(20),
            Customer_Email VARCHAR(150)
        )
    """)

    cursor.execute("""
        CREATE TABLE customerorder (
            Order_Num BIGINT PRIMARY KEY,
            Customer_Num VARCHAR(20),
            Order_Date DATE,
            Order_ReqShipDate DATE,
            Order_ActualShipDate DATE,
            Order_Carrier VARCHAR(100),

            FOREIGN KEY (Customer_Num)
                REFERENCES customer(Customer_Num)
        )
    """)

print("Both MySQL tables created successfully!")

Both MySQL tables created successfully!


In [9]:
customer_columns = list(customers.columns)
order_columns = list(orders.columns)

with connection.cursor() as cursor:

    customer_sql = f"""
        INSERT INTO customer
        ({", ".join(customer_columns)})
        VALUES ({", ".join(["%s"] * len(customer_columns))})
    """

    customer_values = [
        tuple(None if pd.isna(v) else v for v in row)
        for row in customers.itertuples(index=False, name=None)
    ]

    cursor.executemany(customer_sql, customer_values)

    order_sql = f"""
        INSERT INTO customerorder
        ({", ".join(order_columns)})
        VALUES ({", ".join(["%s"] * len(order_columns))})
    """

    order_values = [
        tuple(None if pd.isna(v) else v for v in row)
        for row in orders.itertuples(index=False, name=None)
    ]

    cursor.executemany(order_sql, order_values)

    cursor.execute("SELECT COUNT(*) FROM customer")
    customer_count = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM customerorder")
    order_count = cursor.fetchone()[0]

print("Customer records:", customer_count)
print("CustomerOrder records:", order_count)

Customer records: 10
CustomerOrder records: 9


In [10]:
##Part 1 Question 2
r.flushdb()

for _, row in customers.iterrows():
    customer_id = str(row["Customer_Num"])

    r.hset(
        f"customer:{customer_id}",
        mapping={col: str(row[col]) for col in customers.columns}
    )

    r.sadd("customers:all", customer_id)

for _, row in orders.iterrows():
    order_id = str(row["Order_Num"])
    customer_id = str(row["Customer_Num"])

    r.hset(
        f"order:{order_id}",
        mapping={col: str(row[col]) for col in orders.columns}
    )

    r.sadd(f"orders:customer:{customer_id}", order_id)

    if customer_id in r.smembers("customers:all"):
        r.sadd("customers:with_orders", customer_id)

print("Redis customers:", r.scard("customers:all"))
print("Redis customers with orders:", r.scard("customers:with_orders"))
print("Orders for C25:", r.smembers("orders:customer:C25"))

Redis customers: 10
Redis customers with orders: 7
Orders for C25: {'16644074', '16642826'}


In [11]:
##Part 2 Question 1

with connection.cursor() as cursor:
    cursor.execute("""
        SELECT Customer_City
        FROM customer
        WHERE Customer_Num = 'C25'
    """)
    sql_city = cursor.fetchone()[0]

redis_city = r.hget("customer:C25", "Customer_City")

print("SQL result:", sql_city)
print("Redis result:", redis_city)

SQL result: HERSHEY
Redis result: HERSHEY


In [12]:
##Part 2 Question 2

with connection.cursor() as cursor:
    cursor.execute("""
        SELECT COUNT(*)
        FROM customerorder
        WHERE Customer_Num = 'C25'
    """)
    sql_orders = cursor.fetchone()[0]

redis_orders = r.scard("orders:customer:C25")

print("SQL order count:", sql_orders)
print("Redis order count:", redis_orders)

SQL order count: 2
Redis order count: 2


In [13]:
## Part 2 Question 3

with connection.cursor() as cursor:
    cursor.execute("""
        SELECT Customer_Num, Customer_FirstName,
               Customer_LastName, Customer_City
        FROM customer
        WHERE Customer_State = 'PA'
        ORDER BY Customer_Num
    """)
    sql_pa = cursor.fetchall()

print("SQL Pennsylvania customers:")
for row in sql_pa:
    print(row)

print("\nRedis Pennsylvania customers:")

redis_pa = []
hashes_read = 0

for customer_id in sorted(r.smembers("customers:all")):
    customer = r.hgetall(f"customer:{customer_id}")
    hashes_read += 1

    if customer["Customer_State"] == "PA":
        redis_pa.append((
            customer_id,
            customer["Customer_FirstName"],
            customer["Customer_LastName"],
            customer["Customer_City"]
        ))

for row in redis_pa:
    print(row)

print("\nRedis customer hashes read:", hashes_read)

assert sql_pa == tuple(redis_pa) or list(sql_pa) == redis_pa

SQL Pennsylvania customers:
('C25', 'Bob', 'Felton', 'HERSHEY')
('C2501', 'John', 'Grant', 'PITTSBURGH')

Redis Pennsylvania customers:
('C25', 'Bob', 'Felton', 'HERSHEY')
('C2501', 'John', 'Grant', 'PITTSBURGH')

Redis customer hashes read: 10


In [ ]:
##We use the WHERE clause in SQL whereas in Redis we store customers as hashes and in sets. Retrieved those sets and read the hashes to find the state. We had already created a set in Redis for that custoemer's order numbers. 

In [14]:
##Part 3 
from pathlib import Path

reactor_loader = DATA_DIR / "load_reactor.py"

print("Loader exists:", reactor_loader.exists())
print("CSV exists:", (DATA_DIR / "reactor_sample_5k.csv").exists())

if reactor_loader.exists():
    print(reactor_loader.read_text()[:6000])

Loader exists: False
CSV exists: False
